# Ethiopian Smallholder Crop-Yield Challenge
## Deliverable A - Data Cleaning & Integration Pipeline

**Team:** Team 11  
**Program:** Qiyas / IADE Training Program - Addis Ababa University Hackathon 2026

---

### Objective

The objective of this notebook is to build a robust, reproducible, and leakage-free data integration pipeline for the Ethiopian Smallholder Crop-Yield Challenge.

The pipeline combines plot-level agricultural data with regional weather observations and market-price information. It cleans and standardizes the raw datasets, constructs growing-season weather features, validates the joins, engineers model-ready features, performs automated integrity checks, and exports the final master datasets.

### Final Outputs

This pipeline produces:

- `data/processed/master_train.csv`
- `data/processed/master_test.csv`
- `data/processed/data_dictionary_master.csv`

> **Leakage Prevention:** All statistics required for imputation or preprocessing are learned from the training data only and then applied to the test data.

## 0. Environment Setup & Data Ingestion

This section initializes the Python environment, imports the reusable cleaning and feature-engineering functions, and loads the raw datasets.

The raw files are never manually modified. All transformations are performed programmatically to ensure reproducibility.

The plot training data is treated as the source for any learned preprocessing statistics, preventing information leakage from the leaderboard test dataset.

In [21]:
# ============================================================
# 0. ENVIRONMENT SETUP & DATA INGESTION
# ============================================================

# ------------------------------------------------------------
# 0.1 Import Core Libraries
# ------------------------------------------------------------

import os
import sys
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0.2 Configure Project Imports
# ------------------------------------------------------------

# Add the project root directory so modules inside src/
# can be imported from this notebook.
sys.path.append(os.path.abspath(".."))


# ------------------------------------------------------------
# 0.3 Import Project Cleaning Functions
# ------------------------------------------------------------

from src.cleaning import (
    clean_plot_data,
    clean_weather_data,
    clean_price_data,
    validate_master_tables,
    ALLOWED_REGIONS,
    ALLOWED_CROPS
)


# ------------------------------------------------------------
# 0.4 Import Feature Engineering Functions
# ------------------------------------------------------------

from src.features import (
    build_master_table,
    get_growing_season_months
)


# ------------------------------------------------------------
# 0.5 Define Data Paths
# ------------------------------------------------------------

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"


# ------------------------------------------------------------
# 0.6 Load Raw Datasets
# ------------------------------------------------------------

raw_train = pd.read_csv(
    f"{RAW_DIR}/crop_yield_train.csv"
)

raw_test = pd.read_csv(
    f"{RAW_DIR}/crop_yield_leaderboard_test.csv"
)

raw_weather = pd.read_csv(
    f"{RAW_DIR}/regional_weather.csv"
)

raw_price = pd.read_csv(
    f"{RAW_DIR}/market_prices.csv"
)


# ------------------------------------------------------------
# 0.7 Verify Dataset Dimensions
# ------------------------------------------------------------

print("=== RAW DATASET DIMENSIONS ===")

print(f"Training data:      {raw_train.shape}")
print(f"Leaderboard test:   {raw_test.shape}")
print(f"Regional weather:   {raw_weather.shape}")
print(f"Market prices:      {raw_price.shape}")

=== RAW DATASET DIMENSIONS ===
Training data:      (15090, 15)
Leaderboard test:   (3750, 14)
Regional weather:   (232, 6)
Market prices:      (100, 4)


## A1. Data Cleaning Log

Before integrating the datasets, we document the major data-quality problems identified during profiling.

For every issue, the cleaning log records:

- Source file
- Affected column(s)
- Issue type
- Number and percentage of affected observations
- Cleaning action
- Justification for the chosen treatment

The cleaning strategy focuses on preserving valid observations while correcting inconsistent labels, sentinel values, missing values, duplicate records, and inconsistent price units.

All imputation statistics used for plot-level variables are derived from the training dataset only.

In [22]:
# ============================================================
# A1. DATA CLEANING LOG
# ============================================================

# ------------------------------------------------------------
# A1.1 Document Identified Data-Quality Issues
# ------------------------------------------------------------

cleaning_log_data = [

    # --------------------------------------------------------
    # Plot Data — Region Labels
    # --------------------------------------------------------
    {
        "File": "crop_yield_train.csv",
        "Column(s)": "region",
        "Issue Type": "Case & Whitespace Inconsistencies",
        "Affected Rows (Count)": "4,482",
        "Affected Rows (%)": "29.7%",
        "Fix Applied":
            "Strip whitespace, normalize case, and map "
            "labels to canonical region names.",
        "Justification":
            "Prevents join failures and category fragmentation "
            "across datasets."
    },

    # --------------------------------------------------------
    # Plot Data — Crop Labels
    # --------------------------------------------------------
    {
        "File": "crop_yield_train.csv",
        "Column(s)": "crop_type",
        "Issue Type": "Typos, Whitespace & Mixed Case",
        "Affected Rows (Count)": "5,120",
        "Affected Rows (%)": "33.9%",
        "Fix Applied":
            "Strip whitespace, normalize case, and correct "
            "known crop-name variations such as 'Tef' -> 'teff'.",
        "Justification":
            "Creates consistent crop categories across plot "
            "and market-price datasets."
    },

    # --------------------------------------------------------
    # Plot Data — Sentinel Values
    # --------------------------------------------------------
    {
        "File": "crop_yield_train.csv",
        "Column(s)": "pest_disease_flag, labor_days_per_ha",
        "Issue Type": "Sentinel Values (-999)",
        "Affected Rows (Count)": "788 (300 pest, 488 labor)",
        "Affected Rows (%)": "5.2%",
        "Fix Applied":
            "Replace -999 with NaN; impute pest flag with "
            "training mode and labor with training median.",
        "Justification":
            "The -999 values represent missing information and "
            "would distort numerical analysis if treated as real values."
    },

    # --------------------------------------------------------
    # Plot Data — Missing Numerical Values
    # --------------------------------------------------------
    {
        "File": "crop_yield_train.csv",
        "Column(s)":
            "rainfall_mm_season, soil_quality_index, "
            "fertilizer_kg_per_ha",
        "Issue Type": "Missing Values (NaN)",
        "Affected Rows (Count)": "1,805 total nulls",
        "Affected Rows (%)": "12.0%",
        "Fix Applied":
            "Impute using training-set medians: rainfall 848.76 mm, "
            "soil quality 0.565, fertilizer 33.83 kg/ha.",
        "Justification":
            "Median imputation is robust to skewed distributions, "
            "and training-only statistics prevent leakage."
    },

    # --------------------------------------------------------
    # Weather Data — Region Labels
    # --------------------------------------------------------
    {
        "File": "regional_weather.csv",
        "Column(s)": "region",
        "Issue Type": "Regional Abbreviations & Whitespace",
        "Affected Rows (Count)": "56",
        "Affected Rows (%)": "24.1%",
        "Fix Applied":
            "Map regional abbreviations such as AMH, ORO, SNNP, "
            "SOM, and TIG to canonical region names.",
        "Justification":
            "Standardized region keys are required for reliable "
            "plot-to-weather joins."
    },

    # --------------------------------------------------------
    # Weather Data — Duplicate Monthly Records
    # --------------------------------------------------------
    {
        "File": "regional_weather.csv",
        "Column(s)": "(region, year, month)",
        "Issue Type": "Duplicate Monthly Readings",
        "Affected Rows (Count)": "6 pairs (12 rows total)",
        "Affected Rows (%)": "5.2%",
        "Fix Applied":
            "Remove duplicate records using the standardized "
            "(region, year, month) key.",
        "Justification":
            "Duplicate weather observations could multiply plot "
            "rows during integration and bias seasonal aggregates."
    },

    # --------------------------------------------------------
    # Market Prices — Unit Inconsistency
    # --------------------------------------------------------
    {
        "File": "market_prices.csv",
        "Column(s)": "price_birr_per_quintal",
        "Issue Type": "Unit Scale Mix-up (Birr/kg vs Birr/quintal)",
        "Affected Rows (Count)": "4",
        "Affected Rows (%)": "4.0%",
        "Fix Applied":
            "Multiply verified Birr/kg observations below the "
            "identified threshold by 100 to convert to Birr/quintal.",
        "Justification":
            "One quintal equals 100 kg, so all prices must use "
            "a common Birr-per-quintal unit."
    },

    # --------------------------------------------------------
    # Market Prices — Missing Values
    # --------------------------------------------------------
    {
        "File": "market_prices.csv",
        "Column(s)": "price_birr_per_quintal",
        "Issue Type": "Missing Values (NaN)",
        "Affected Rows (Count)": "4",
        "Affected Rows (%)": "4.0%",
        "Fix Applied":
            "Impute using the median price for the corresponding crop.",
        "Justification":
            "Maintains a complete price reference table for "
            "economic and revenue analysis."
    }
]


# ------------------------------------------------------------
# A1.2 Convert Cleaning Log to DataFrame
# ------------------------------------------------------------

df_cleaning_log = pd.DataFrame(cleaning_log_data)


# ------------------------------------------------------------
# A1.3 Display Cleaning Log
# ------------------------------------------------------------

display(df_cleaning_log)

,File,Column(s),Issue Type,Affected Rows (Count),Affected Rows (%),Fix Applied,Justification
0,crop_yield_train.csv,region,Case & Whitespace Inconsistencies,"4,482",29.7%,"Strip whitespace, normalize case, and map labe...",Prevents join failures and category fragmentat...
1,crop_yield_train.csv,crop_type,"Typos, Whitespace & Mixed Case","5,120",33.9%,"Strip whitespace, normalize case, and correct ...",Creates consistent crop categories across plot...
2,crop_yield_train.csv,"pest_disease_flag, labor_days_per_ha",Sentinel Values (-999),"788 (300 pest, 488 labor)",5.2%,Replace -999 with NaN; impute pest flag with t...,The -999 values represent missing information ...
3,crop_yield_train.csv,"rainfall_mm_season, soil_quality_index, fertil...",Missing Values (NaN),"1,805 total nulls",12.0%,Impute using training-set medians: rainfall 84...,Median imputation is robust to skewed distribu...
4,regional_weather.csv,region,Regional Abbreviations & Whitespace,56,24.1%,"Map regional abbreviations such as AMH, ORO, S...",Standardized region keys are required for reli...
5,regional_weather.csv,"(region, year, month)",Duplicate Monthly Readings,6 pairs (12 rows total),5.2%,Remove duplicate records using the standardize...,Duplicate weather observations could multiply ...
6,market_prices.csv,price_birr_per_quintal,Unit Scale Mix-up (Birr/kg vs Birr/quintal),4,4.0%,Multiply verified Birr/kg observations below t...,"One quintal equals 100 kg, so all prices must ..."
7,market_prices.csv,price_birr_per_quintal,Missing Values (NaN),4,4.0%,Impute using the median price for the correspo...,Maintains a complete price reference table for...


## A2. Key Standardization Proof

Reliable integration requires identical categorical keys across all source tables.

This section demonstrates the unique values of:

- `region` across plot, weather, and market-price datasets
- `crop_type` across plot and market-price datasets

We first display the raw values, apply the reusable cleaning pipeline, and then verify that all datasets use the same canonical categories.

The test dataset receives the preprocessing statistics learned from the training dataset rather than calculating its own values.

In [23]:
# ============================================================
# A2. KEY STANDARDIZATION PROOF
# ============================================================

# ------------------------------------------------------------
# A2.1 Display Raw Region Categories
# ------------------------------------------------------------

print("=== BEFORE CLEANING: UNIQUE REGION VALUES ===")

print(
    "Plot Train Regions:",
    sorted([str(x) for x in raw_train["region"].unique()])
)

print(
    "Weather Regions:",
    sorted([str(x) for x in raw_weather["region"].unique()])
)

print(
    "Price Regions:",
    sorted([str(x) for x in raw_price["region"].unique()])
)


# ------------------------------------------------------------
# A2.2 Display Raw Crop Categories
# ------------------------------------------------------------

print("\n=== BEFORE CLEANING: UNIQUE CROP VALUES ===")

print(
    "Plot Train Crops:",
    sorted([str(x) for x in raw_train["crop_type"].unique()])
)

print(
    "Price Crops:",
    sorted([str(x) for x in raw_price["crop_type"].unique()])
)


# ------------------------------------------------------------
# A2.3 Clean Plot Data
# ------------------------------------------------------------

# Learn preprocessing statistics exclusively from training data.
clean_train, learned_stats = clean_plot_data(
    raw_train,
    is_train=True
)

# Apply the same learned statistics to leaderboard test data.
clean_test, _ = clean_plot_data(
    raw_test,
    is_train=False,
    stats=learned_stats
)


# ------------------------------------------------------------
# A2.4 Clean Weather and Market-Price Data
# ------------------------------------------------------------

clean_weather = clean_weather_data(raw_weather)
clean_price = clean_price_data(raw_price)


# ------------------------------------------------------------
# A2.5 Display Standardized Region Categories
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("=== AFTER CLEANING: STANDARDIZED REGIONS ===")

print(
    "Plot Train:",
    sorted(clean_train["region"].unique())
)

print(
    "Plot Test:",
    sorted(clean_test["region"].unique())
)

print(
    "Weather:",
    sorted(clean_weather["region"].unique())
)

print(
    "Prices:",
    sorted(clean_price["region"].unique())
)


# ------------------------------------------------------------
# A2.6 Display Standardized Crop Categories
# ------------------------------------------------------------

print("\n=== AFTER CLEANING: STANDARDIZED CROPS ===")

print(
    "Plot Train:",
    sorted(clean_train["crop_type"].unique())
)

print(
    "Plot Test:",
    sorted(clean_test["crop_type"].unique())
)

print(
    "Prices:",
    sorted(clean_price["crop_type"].unique())
)


# ------------------------------------------------------------
# A2.7 Verify Canonical Categories
# ------------------------------------------------------------

assert set(clean_train["region"]) == set(ALLOWED_REGIONS)
assert set(clean_test["region"]) == set(ALLOWED_REGIONS)
assert set(clean_weather["region"]) == set(ALLOWED_REGIONS)
assert set(clean_price["region"]) == set(ALLOWED_REGIONS)

assert set(clean_train["crop_type"]) == set(ALLOWED_CROPS)
assert set(clean_test["crop_type"]) == set(ALLOWED_CROPS)
assert set(clean_price["crop_type"]) == set(ALLOWED_CROPS)

print("\nPASS: All datasets share identical canonical labels.")

=== BEFORE CLEANING: UNIQUE REGION VALUES ===
Plot Train Regions: ['AMHARA', 'Amhara', 'Amhara ', 'OROMIA', 'Oromia', 'Oromia ', 'SNNPR', 'SNNPR ', 'SOMALI', 'Somali', 'Somali ', 'TIGRAY', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']
Weather Regions: ['AMH', 'Amhara', 'Amhara ', 'ORO', 'Oromia', 'Oromia ', 'SNNP', 'SNNPR', 'SNNPR ', 'SOM', 'Somali', 'Somali ', 'TIG', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']
Price Regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']

=== BEFORE CLEANING: UNIQUE CROP VALUES ===
Plot Train Crops: [' barley ', ' maize ', ' sorghum ', ' teff ', ' wheat ', 'BARLEY', 'Barley', 'MAIZE', 'Maize', 'SORGHUM', 'Sorghum', 'TEFF', 'Tef', 'Teff', 'WHEAT', 'Wheat', 'barley', 'maize', 'sorghum', 'teff', 'wheat']
Price Crops: [' barley ', ' maize ', ' sorghum ', ' teff ', ' wheat ', 'Barley', 'Maize', 'Sorghum', 'Wheat', 'barley', 'maize', 'sorghum', 'teff', 'wheat']

=== AFTER CLEANING: STANDARDIZED REGIONS ==

## A3. Join Map & Integration Architecture

The cleaned plot dataset serves as the **left table** throughout integration because every plot must remain in the final master dataset.

### Weather Integration

Weather data is available monthly by `region`, `year`, and `month`. For each plot, we define its growing season as:

> **Planting month + the following 3 consecutive months**

This creates a **4-month growing-season window**.

The monthly weather observations are aggregated into seasonal temperature, rainfall, and extreme-heat features before being attached to each plot.

### Market-Price Integration

Market prices are matched using:

- `crop_type`
- `region`
- `survey_year`

Market price is retained for economic analysis and revenue estimation and is not intended to serve as a crop-yield prediction feature.

### Join Strategy

Both integrations use left joins to preserve every training and leaderboard-test plot.

```mermaid
graph TD

    P[Plot Data<br/>Train / Test<br/>Left Table]

    W1[Monthly Regional Weather]
    W2[Clean & Standardize Weather]
    W3[4-Month Growing-Season Aggregation]

    M1[Market Prices]
    M2[Clean Units & Missing Values]

    J1[Plot + Weather]
    J2[Final Master Table]

    W1 --> W2
    W2 --> W3

    M1 --> M2

    P -->|Left Join| J1
    W3 -->|region + year + planting month| J1

    J1 -->|Left Join| J2
    M2 -->|crop + region + year| J2


One thing to fix in your challenge-facing wording: call this a **4-month growing-season window**, because the current implementation explicitly defines planting month plus three subsequent months. :chatgpt-content-reference{index="4"}

---

# 6. A4 — Markdown cell

```markdown
## A4. Join Audit

After constructing the integrated master tables, we verify that the joins preserve the original plot populations and correctly attach external information.

The audit evaluates:

- Weather match rate
- Market-price match rate
- Training row count before and after integration
- Test row count before and after integration
- Number of weather months available within each growing-season window
- Treatment of incomplete weather windows

A successful many-to-one left join should enrich each plot without increasing or decreasing the number of plot records.

In [25]:
# ============================================================
# A4. JOIN AUDIT
# ============================================================

# ------------------------------------------------------------
# A4.1 Build Integrated Master Tables
# ------------------------------------------------------------

master_train = build_master_table(
    clean_train,
    clean_weather,
    clean_price
)

master_test = build_master_table(
    clean_test,
    clean_weather,
    clean_price
)


# ------------------------------------------------------------
# A4.2 Verify Row Preservation
# ------------------------------------------------------------

print("=== ROW COUNT AUDIT ===")

print(
    f"Train rows before: {len(clean_train):,} | "
    f"after: {len(master_train):,} | "
    f"delta: {len(master_train) - len(clean_train):,}"
)

print(
    f"Test rows before:  {len(clean_test):,} | "
    f"after: {len(master_test):,} | "
    f"delta: {len(master_test) - len(clean_test):,}"
)


# ------------------------------------------------------------
# A4.3 Calculate Actual Weather Match Rate
# ------------------------------------------------------------

weather_matched = master_train[
    "weather_season_mean_temp"
].notna()

weather_match_rate = weather_matched.mean() * 100

print(
    f"\nWeather match rate: "
    f"{weather_match_rate:.2f}%"
)


# ------------------------------------------------------------
# A4.4 Calculate Actual Price Match Rate
# ------------------------------------------------------------

price_matched = master_train[
    "price_birr_per_quintal"
].notna()

price_match_rate = price_matched.mean() * 100

print(
    f"Price match rate: "
    f"{price_match_rate:.2f}%"
)


# ------------------------------------------------------------
# A4.5 Audit Growing-Season Weather Coverage
# ------------------------------------------------------------

print("\n=== WEATHER MONTH COVERAGE ===")

coverage = (
    master_train["weather_months_available"]
    .value_counts()
    .sort_index(ascending=False)
)

for months, count in coverage.items():

    percentage = (
        count / len(master_train)
    ) * 100

    print(
        f"{months} months available: "
        f"{count:,} plots ({percentage:.1f}%)"
    )


# ------------------------------------------------------------
# A4.6 Document Missing-Month Treatment
# ------------------------------------------------------------

print("\nMissing-month handling:")

print(
    "For incomplete 4-month windows, temperature is averaged "
    "over available observations. Rainfall and extreme-heat "
    "totals are proportionally scaled to a four-month equivalent."
)

=== ROW COUNT AUDIT ===
Train rows before: 15,090 | after: 15,090 | delta: 0
Test rows before:  3,750 | after: 3,750 | delta: 0

Weather match rate: 100.00%
Price match rate: 100.00%

=== WEATHER MONTH COVERAGE ===
4 months available: 12,165 plots (80.6%)
3 months available: 2,477 plots (16.4%)
2 months available: 448 plots (3.0%)

Missing-month handling:
For incomplete 4-month windows, temperature is averaged over available observations. Rainfall and extreme-heat totals are proportionally scaled to a four-month equivalent.


## A5. Join Proof - Three Sample Plots

To verify the weather integration at the individual-plot level, three sample plots are traced through the complete growing-season aggregation process.

For each plot, we display:

1. Region, survey year, and planting month
2. Expected 4-month growing-season window
3. Monthly weather observations selected from the weather table
4. Final seasonal weather values stored in the master dataset

This provides row-level evidence that the weather aggregation and join logic operate as intended.

In [26]:
# ============================================================
# A5. JOIN PROOF — THREE SAMPLE PLOTS
# ============================================================

# ------------------------------------------------------------
# A5.1 Select Sample Plots
# ------------------------------------------------------------

sample_plots = [
    "PLOT-000339",
    "PLOT-007457",
    master_train["plot_id"].iloc[100]
]


# ------------------------------------------------------------
# A5.2 Trace Each Plot Through Weather Aggregation
# ------------------------------------------------------------

for plot_id in sample_plots:

    # Retrieve integrated plot record.
    row = master_train[
        master_train["plot_id"] == plot_id
    ].iloc[0]

    region = row["region"]
    year = row["survey_year"]
    planting_month = row["planting_month"]

    # Determine expected growing-season months.
    growing_months = get_growing_season_months(
        planting_month
    )

    # Retrieve underlying monthly weather observations.
    weather_rows = clean_weather[
        (clean_weather["region"] == region)
        & (clean_weather["year"] == year)
        & (clean_weather["month"].isin(growing_months))
    ]

    print("=" * 70)
    print(f"PLOT ID: {plot_id}")
    print(
        f"Region: {region} | "
        f"Year: {year} | "
        f"Planting Month: {planting_month}"
    )

    print(
        f"Expected Growing-Season Months: "
        f"{growing_months}"
    )

    print("\nUnderlying Weather Observations:")

    display(
        weather_rows[
            [
                "month",
                "avg_temp_c",
                "monthly_rainfall_mm",
                "extreme_heat_days"
            ]
        ]
    )

    print(
        f"Season Mean Temperature: "
        f"{row['weather_season_mean_temp']:.2f} °C"
    )

    print(
        f"Season Rainfall: "
        f"{row['weather_season_rainfall_mm']:.2f} mm"
    )

    print(
        f"Season Extreme Heat Days: "
        f"{row['weather_season_heat_days']:.1f}"
    )

    print(
        f"Temperature Anomaly: "
        f"{row['weather_temp_anomaly_c']:.2f} °C"
    )

PLOT ID: PLOT-000339
Region: Oromia | Year: 2024 | Planting Month: Aug
Expected Growing-Season Months: ['Aug', 'Sep', 'Oct', 'Nov']

Underlying Weather Observations:


,month,avg_temp_c,monthly_rainfall_mm,extreme_heat_days
77,Aug,19.1,185.1,0
164,Sep,15.1,147.1,0
187,Nov,19.4,20.8,0


Season Mean Temperature: 17.87 °C
Season Rainfall: 470.67 mm
Season Extreme Heat Days: 0.0
Temperature Anomaly: -1.18 °C
PLOT ID: PLOT-007457
Region: Amhara | Year: 2023 | Planting Month: Jun
Expected Growing-Season Months: ['Jun', 'Jul', 'Aug', 'Sep']

Underlying Weather Observations:


,month,avg_temp_c,monthly_rainfall_mm,extreme_heat_days
83,Jun,17.0,118.5,0
153,Aug,14.5,192.8,0
209,Jul,14.9,244.6,0
224,Sep,14.4,153.5,0


Season Mean Temperature: 15.20 °C
Season Rainfall: 709.40 mm
Season Extreme Heat Days: 0.0
Temperature Anomaly: -1.89 °C
PLOT ID: PLOT-002298
Region: Oromia | Year: 2024 | Planting Month: Jul
Expected Growing-Season Months: ['Jul', 'Aug', 'Sep', 'Oct']

Underlying Weather Observations:


,month,avg_temp_c,monthly_rainfall_mm,extreme_heat_days
38,Jul,17.0,205.7,0
77,Aug,19.1,185.1,0
164,Sep,15.1,147.1,0


Season Mean Temperature: 17.07 °C
Season Rainfall: 717.20 mm
Season Extreme Heat Days: 0.0
Temperature Anomaly: -1.98 °C


## A6. Feature Engineering

The integrated data is transformed into additional domain-informed predictors designed to capture climate exposure, agricultural input interactions, farm-management intensity, and seasonal timing.

The engineered feature set includes:

- At least three weather-derived variables
- Agronomic interaction features
- A rainfall comparison feature
- A farm-management intensity feature
- A planting-season indicator

The table below documents each engineered feature, its formula, source variables, and agronomic rationale.

In [27]:
features_table = [
    {
        'Feature Name': 'weather_season_mean_temp',
        'Formula': 'Mean(avg_temp_c) over 4-month window',
        'Source Columns': 'avg_temp_c, planting_month',
        'Agronomic Rationale': 'Captures thermal requirements and heat accumulation essential for crop phenology.'
    },
    {
        'Feature Name': 'weather_season_rainfall_mm',
        'Formula': '(Sum(monthly_rainfall_mm) / k) * 4',
        'Source Columns': 'monthly_rainfall_mm, planting_month',
        'Agronomic Rationale': 'Measures cumulative meteorological moisture available over the active vegetative and grain-filling period.'
    },
    {
        'Feature Name': 'weather_season_heat_days',
        'Formula': '(Sum(extreme_heat_days) / k) * 4',
        'Source Columns': 'extreme_heat_days, planting_month',
        'Agronomic Rationale': 'Extreme temperature spikes cause pollen sterility and severe yield penalties.'
    },
    {
        'Feature Name': 'weather_temp_anomaly_c',
        'Formula': 'season_mean_temp - 4_year_regional_baseline_temp',
        'Source Columns': 'avg_temp_c, region',
        'Agronomic Rationale': 'Isolates regional thermal shocks relative to historical climatic norms.'
    },
    {
        'Feature Name': 'fertilizer_improved_seed_interaction',
        'Formula': 'fertilizer_kg_per_ha * improved_seed_used',
        'Source Columns': 'fertilizer_kg_per_ha, improved_seed_used',
        'Agronomic Rationale': 'Certified improved seeds have superior genetic potential and higher marginal response to nutrient inputs.'
    },
    {
        'Feature Name': 'weather_rain_discrepancy_ratio',
        'Formula': 'rainfall_mm_season / (weather_season_rainfall_mm + 1e-5)',
        'Source Columns': 'rainfall_mm_season, weather_season_rainfall_mm',
        'Agronomic Rationale': 'Identifies microclimatic discrepancies between plot-level farmer perception and regional station data.'
    },
    {
        'Feature Name': 'labor_intensity_per_farm_size',
        'Formula': 'labor_days_per_ha / (farm_size_ha + 0.1)',
        'Source Columns': 'labor_days_per_ha, farm_size_ha',
        'Agronomic Rationale': 'Measures labor density and management intensity per farm unit.'
    },
    {
        'Feature Name': 'is_meher_season',
        'Formula': '1 if planting_month in [Jun, Jul, Aug] else 0',
        'Source Columns': 'planting_month',
        'Agronomic Rationale': 'Differentiates Ethiopia main rainy season (Meher) from short secondary season (Belg).'
    }
]

df_features = pd.DataFrame(features_table)
display(df_features)


,Feature Name,Formula,Source Columns,Agronomic Rationale
0,weather_season_mean_temp,Mean(avg_temp_c) over 4-month window,"avg_temp_c, planting_month",Captures thermal requirements and heat accumul...
1,weather_season_rainfall_mm,(Sum(monthly_rainfall_mm) / k) * 4,"monthly_rainfall_mm, planting_month",Measures cumulative meteorological moisture av...
2,weather_season_heat_days,(Sum(extreme_heat_days) / k) * 4,"extreme_heat_days, planting_month",Extreme temperature spikes cause pollen steril...
3,weather_temp_anomaly_c,season_mean_temp - 4_year_regional_baseline_temp,"avg_temp_c, region",Isolates regional thermal shocks relative to h...
4,fertilizer_improved_seed_interaction,fertilizer_kg_per_ha * improved_seed_used,"fertilizer_kg_per_ha, improved_seed_used",Certified improved seeds have superior genetic...
5,weather_rain_discrepancy_ratio,rainfall_mm_season / (weather_season_rainfall_...,"rainfall_mm_season, weather_season_rainfall_mm",Identifies microclimatic discrepancies between...
6,labor_intensity_per_farm_size,labor_days_per_ha / (farm_size_ha + 0.1),"labor_days_per_ha, farm_size_ha",Measures labor density and management intensit...
7,is_meher_season,"1 if planting_month in [Jun, Jul, Aug] else 0",planting_month,Differentiates Ethiopia main rainy season (Meh...


## A7. Automated Integrity Checks

Before exporting the processed datasets, automated validation checks are executed to confirm that the integration pipeline has produced structurally valid and model-ready data.

The validation suite checks critical properties such as:

- Plot-ID uniqueness
- Expected row counts
- Valid categorical domains
- Train/test schema alignment
- Required weather-derived features
- Absence of prohibited leakage
- Valid target and feature ranges

Any failed critical check raises an error and prevents the pipeline from silently exporting invalid master datasets.

In [28]:
# ============================================================
# A7. AUTOMATED INTEGRITY CHECKS
# ============================================================

# ------------------------------------------------------------
# A7.1 Execute Validation Suite
# ------------------------------------------------------------

audit_results = validate_master_tables(
    master_train,
    master_test
)


# ------------------------------------------------------------
# A7.2 Display PASS / FAIL Report
# ------------------------------------------------------------

print("=" * 70)
print("AUTOMATED INTEGRITY VALIDATION")
print("=" * 70)

all_pass = True

for check_name, result in audit_results.items():

    status = result["status"]
    detail = result["detail"]

    if status != "PASS":
        all_pass = False

    print(
        f"[{status}] "
        f"{check_name} -> {detail}"
    )

print("=" * 70)


# ------------------------------------------------------------
# A7.3 Stop Pipeline if Validation Fails
# ------------------------------------------------------------

if all_pass:

    print(
        "PASS: All automated integrity checks completed "
        "successfully."
    )

else:

    raise AssertionError(
        "One or more integrity checks failed. "
        "Review the validation report before exporting."
    )

AUTOMATED INTEGRITY VALIDATION
[PASS] 1_plot_id_uniqueness -> Train unique: 15090/15090, Test unique: 3750/3750
[PASS] 2_row_count_invariance -> Train rows: 15090 (expected 15090), Test rows: 3750 (expected 3750)
[PASS] 3_zero_missing_values -> Train feature nulls: 0, Test feature nulls: 0
[PASS] 4_canonical_labels -> Regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray'], Crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']
[PASS] 5_physical_ranges -> yield >= 0, farm_size > 0, price > 0, fertilizer >= 0
[PASS] 6_schema_alignment -> Features aligned: 25 test cols vs 26 train cols
PASS: All automated integrity checks completed successfully.


## A8. Master Table & Data Dictionary Export

After all cleaning, integration, feature-engineering, and integrity checks pass, the final processed datasets are exported to `data/processed/`.

The required outputs are:

- `master_train.csv`
- `master_test.csv`
- `data_dictionary_master.csv`

The exported master datasets will serve as the common source for exploratory analysis, visualization, modeling, and deployment.

No manual editing is performed after export.

In [30]:
# ============================================================
# A8. MASTER TABLE & DATA DICTIONARY EXPORT
# ============================================================

# ------------------------------------------------------------
# A8.1 Ensure Output Directory Exists
# ------------------------------------------------------------

os.makedirs(
    PROCESSED_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# A8.2 Define Output Paths
# ------------------------------------------------------------

master_train_path = (
    f"{PROCESSED_DIR}/master_train.csv"
)

master_test_path = (
    f"{PROCESSED_DIR}/master_test.csv"
)

dictionary_path = (
    f"{PROCESSED_DIR}/data_dictionary_master.csv"
)


# ------------------------------------------------------------
# A8.3 Export Master Datasets
# ------------------------------------------------------------

master_train.to_csv(
    master_train_path,
    index=False
)

master_test.to_csv(
    master_test_path,
    index=False
)


# ------------------------------------------------------------
# A8.4 Verify Required Exports
# ------------------------------------------------------------

print("=== EXPORT VERIFICATION ===")

print(
    f"master_train.csv: "
    f"{os.path.exists(master_train_path)} "
    f"{master_train.shape}"
)

print(
    f"master_test.csv: "
    f"{os.path.exists(master_test_path)} "
    f"{master_test.shape}"
)

print(
    f"data_dictionary_master.csv: "
    f"{os.path.exists(dictionary_path)}"
)


# ------------------------------------------------------------
# A8.5 Preview Final Master Dataset
# ------------------------------------------------------------

display(master_train.head(3))

=== EXPORT VERIFICATION ===
master_train.csv: True (15090, 26)
master_test.csv: True (3750, 25)
data_dictionary_master.csv: True


,plot_id,region,crop_type,survey_year,planting_month,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,...,weather_season_rainfall_mm,weather_season_heat_days,weather_temp_anomaly_c,weather_months_available,price_birr_per_quintal,fertilizer_improved_seed_interaction,weather_rain_discrepancy_ratio,labor_intensity_per_farm_size,is_meher_season,altitude_temp_index
0,PLOT-000339,Oromia,barley,2024,Aug,2697.817349,1157.403560,0.867604,29.393684,0,...,470.666667,0.0,-1.178986,3,4590.0,0.000000,2.459073,44.106971,1,9.270319
1,PLOT-007457,Amhara,maize,2023,Jun,1523.350141,897.420122,1.099197,16.843992,1,...,709.400000,0.0,-1.890000,4,2979.0,16.843992,1.265041,37.834071,1,5.282990
2,PLOT-006092,Oromia,wheat,2024,Feb,2389.441948,1119.155741,0.811309,41.015144,0,...,294.900000,0.0,0.154348,4,5387.0,0.000000,3.795035,57.989247,0,8.173224
